# 1. Import the necessary libraries

In [1]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_problem
from problems.microgrid_function import microgrid_function
from tuners import fine_tune_mesh, fine_tune_mesh_old, fine_tune_nsga2

from contextlib import redirect_stdout
from io import StringIO
from joblib import Parallel, delayed
from optuna.pruners import NopPruner
from pymoo.indicators.hv import HV

import numpy as np

# 2. Fine tuning configuration by hypervolume

In [2]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 22

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [('Lead_Acid', 3, 3, None), ('Li-ion', 3, 3, None), ('ZEBRA', 3, 3, None), ('NaS', 3, 3, None), ('NiCd', 3, 3, None), ('NiMH', 3, 3, None), ('RFV', 3, 3, None), ('ZnBr', 3, 3, None)]

# Algorithm configuration
population_size = 100 # Population size
max_iteration = None # Number of iterations/generations
max_fitness_eval = 5000 # Maximum number of fitness evaluations
random_state = None # Set a seed for random numbers (not used if it is None)

tuning_folder = '../hyperparams' # Folder to get the tuned parameters

# Fine tuning configuration
n_trials = 30 # Number of trials for Optuna
n_steps = 5 # Number of rounds in a trial
optuna_pruner = NopPruner() # Optuna's pruner to prune bad trials according to a metric
# optuna_pruner = optuna.pruners.WilcoxonPruner(p_threshold=0.05, n_startup_steps=n_steps//2)
hypercube_vertex = 10 # Consider the vertex of the hypercube
######################################################

ref_point = [np.array([hypercube_vertex] * exp[1]) for exp in experiments] # Reference point for hypervolume calculation
indicators = [HV(ref_point=rf) for rf in ref_point] # Define the indicator to calculate the volume

# 3 Define auxiliar functions

In [3]:
def get_fitness_function(func_name, objective_dim, position_dim, wfg_k):
	select_bat = {'Lead_Acid': 0, 'Li-ion': 1, 'ZEBRA': 2, 'NaS': 3, 'NiCd': 4, 'NiMH': 5, 'RFV': 6, 'ZnBr': 7}
	# Microgrid problem
	if func_name in select_bat:
		load = np.genfromtxt('../seasonal_data/load.txt')
		temperature = np.genfromtxt('../seasonal_data/temperature.txt')
		solar_data = np.genfromtxt('../seasonal_data/irradiance.txt')
		wind_data = np.genfromtxt('../seasonal_data/wind.txt')
		func = lambda args: microgrid_function(args[0], args[1], args[2], select_bat[func_name], load, temperature, solar_data, wind_data)
		position_min_value = np.array([1.0, 1.0, 1.0])
		position_max_value = np.array([2000.0, 2000.0, 2000.0])
	# Benchmark problems
	else:
		func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
	return func, objective_dim, position_dim, position_min_value, position_max_value

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = StringIO()
		with redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 3. Apply fine tuning in the algorithms (in parallel)

## 3.1 Tuning MESH

In [4]:
#################### CUSTOMIZABLE ####################

# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
mesh_global_best_type = [0,1] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0,1,2] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0,1,2,3,4] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)
######################################################

In [5]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
     (n_trials, n_steps, optuna_pruner),
	 get_fitness_function(exp[0], exp[1], exp[2], exp[3]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(fine_tune_mesh, params) for params in params_list)

[W 2026-03-02 17:22:14,068] Trial 0 failed with parameters: {'communication_probability': 0.3565664865011372, 'mutation_rate': 0.9206876919433741, 'personal_guide_array_size': 3} because of the following error: TypeError('only length-1 arrays can be converted to Python scalars').
Traceback (most recent call last):
  File "/home/leonardo_filho/miniconda3/envs/mesh/lib/python3.13/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/home/leonardo_filho/Área de trabalho/Mesh-python/scripts/notebooks/tuners.py", line 78, in tuning
    mesh.run()
    ~~~~~~~~^^
  File "/home/leonardo_filho/Área de trabalho/Mesh-python/src/mesh/core.py", line 503, in run
    self.initialize()
    ~~~~~~~~~~~~~~~^^
  File "/home/leonardo_filho/Área de trabalho/Mesh-python/src/mesh/core.py", line 178, in initialize
    self.population.fitness[:] = self.evaluate(self.population.position)
                                 ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^

['Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scalars',
 'Error: only length-1 arrays can be converted to Python scala

## Tuning MESH (previous implementation)

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'OLD_MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
     (n_trials, n_steps, optuna_pruner),
	 get_fitness_function(exp[0], exp[1], exp[2], exp[3]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(fine_tune_mesh_old, params) for params in params_list)

## 3.2 Tuning NSGA2

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
    (n_trials, n_steps, optuna_pruner),
	 get_fitness_function(exp[0], exp[1], exp[2], exp[3]),
	 (),
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(fine_tune_nsga2, params) for params in params_list)